## Mouse Methylation Atlas Datasets

### GSE290585: Atlas Dataset
- **Publication:** [PMC12506328](https://pmc.ncbi.nlm.nih.gov/articles/PMC12506328/)
- **Dataset:** `GSE290585_SeSaMeBeta_MM285_BS`
- **Purpose:** Primary dataset used to build the **mouse methylation atlas**.

### GSE196902: Independent Validation Dataset
- **Publication:** [PubMed 35297293](https://pubmed.ncbi.nlm.nih.gov/35297293/)
- **Purpose:** Separate, independent **MM285 validation dataset**.
- Used as a **completely independent validation set** for the mouse atlas.
- Enables:
  - Creation of **independent mixtures** for validation.
  - Use of the **full GSE290585 dataset** for atlas construction, without requiring a train/test split.

In [1]:
import sys

import pandas as pd

sys.path.insert(0, "../src")
import config

path = config.DATA_DIR / "raw" / "GSE196902_family.soft"

records = []
current = {}

with open(path, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()

        # New GEO sample
        if line.startswith("^SAMPLE"):
            if current:
                records.append(current)
            current = {"GSM": line.split("=", 1)[1].strip()}

        # Sample title
        elif line.startswith("!Sample_title"):
            current["title"] = line.split("=", 1)[1].strip()

        # Sample metadata
        elif line.startswith("!Sample_characteristics_ch1"):
            value = line.split("=", 1)[1].strip()

            if ": " in value:
                key, val = value.split(": ", 1)
                current[key] = val

# Add final sample
if current:
    records.append(current)

meta = pd.DataFrame(records)

print(meta.shape)
display(meta)


(67, 7)


,GSM,title,tissue,tissue type,biological sex,strain,sample preservation
0,GSM5904451,HAFTL_replicate1,Hematopoietic precursor,Cell line,NaN,NaN,NaN
1,GSM5904452,DNMT1_WT,Embryonic fibroblasts,Cell line,NaN,NaN,NaN
2,GSM5904453,DNMT1_KO,Embryonic fibroblasts,Cell line,NaN,NaN,NaN
3,GSM5904454,C2C12_replicate1,Hematopoietic precursor,Cell line,NaN,NaN,NaN
4,GSM5904455,HAFTL_replicate2,Hematopoietic precursor,Cell line,NaN,NaN,NaN
...,...,...,...,...,...,...,...
62,GSM5904513,P63,Skin,Normal,Male,C57BL/6J & FVB,FF
63,GSM5904514,P64,Lung,Normal,Female,C57BL/6J,FF
64,GSM5904515,P65,Lung,Normal,Female,C57BL/6J,FFPE
65,GSM5904516,P66,Lung,Normal,Female,C57BL/6J,FFPE


In [2]:
normal = meta[meta["tissue type"] == "Normal"]

print(normal.shape)
display(normal[["GSM", "title", "tissue", "biological sex", "strain", "sample preservation"]])

(56, 7)


,GSM,title,tissue,biological sex,strain,sample preservation
11,GSM5904462,P12,Brain,Male,C57BL/6J,FF
12,GSM5904463,Spleen1_replicate1,Spleen,Male,C57BL/6J & 129/Sv,FF
13,GSM5904464,P14,Colon,Male,C57BL/6J,FF
14,GSM5904465,P15,Brain,Male,C57BL/6J,FF
15,GSM5904466,P16,Lung,Male,C57BL/6J & 129/Sv,FF
16,GSM5904467,P17,BM,Male,C57BL/6J,FF
17,GSM5904468,P18,Brain,Female,C57BL/6J,FF
18,GSM5904469,P19,Thymus,Male,C57BL/6J & 129/Sv,FF
19,GSM5904470,P20,Skin,Male,C57BL/6J,FF
20,GSM5904471,P21,Lung,Male,C57BL/6J & 129/Sv,FF


---

## Processing GSE196902 into an atlas-shaped matrix

The raw matrix is not directly usable: its columns are opaque GEO sample titles (`P12`, `P15`,
`Spleen1_replicate1`), and each sample column is followed by a `Detection Pval` column.

The steps below reshape it to match `GSE290585_SeSaMeBeta_MM285_BS_simplified_columns.csv`
(`probe_ID` first, then one tissue-named column per replicate) and write the result to a **new**
file. Both source files are read-only throughout.

| Decision | Value |
|---|---|
| Samples | 56 `Normal` only; the 11 cell lines are dropped |
| Tissue labels | verbatim from the SOFT file (`BM`, `Brain`, `Breast`, `Prostate`, …), **not** harmonised to atlas vocabulary |
| Precision | full source precision, no rounding |
| Probes | all 287,050 retained; intersection with the atlas panel belongs at inference time |


In [3]:
# ---- Build the column plan ---------------------------------------------------
# Decide which columns to keep and what to call them, before loading anything.

data_path = config.DATA_DIR / "raw" / "GSE196902_Matrix_processed.csv"

# title -> tissue, restricted to Normal samples (cell lines are excluded)
title_to_tissue = dict(zip(normal["title"], normal["tissue"]))

# Read the header only: the full matrix is ~277 MB, so decide columns before loading it.
header = pd.read_csv(data_path, nrows=0).columns.tolist()

probe_col   = header[0]                                                # "ID_REF"
pval_cols   = [c for c in header[1:] if c.startswith("Detection Pval")]
sample_cols = [c for c in header[1:] if not c.startswith("Detection Pval")]

# Every Normal title must exist as a column, otherwise the mapping is incomplete.
missing = sorted(set(title_to_tissue) - set(sample_cols))
assert not missing, f"Normal titles absent from the matrix: {missing}"

# Keep Normal samples only, ordered by (tissue, title) so replicates of the same tissue sit
# next to each other, matching the atlas file layout. The sort has to happen here, while the
# columns still carry unique titles; after relabelling, tissue names are duplicated.
keep_cols = sorted(
    (c for c in sample_cols if c in title_to_tissue),
    key=lambda c: (title_to_tissue[c], c),
)

print(f"probe column:        {probe_col}")
print(f"sample columns:      {len(sample_cols)}")
print(f"detection p-value:   {len(pval_cols)}  (dropped)")
print(f"normal samples kept: {len(keep_cols)}  "
      f"({len(sample_cols) - len(keep_cols)} cell lines dropped)")
print(f"distinct tissues:    {len(set(title_to_tissue.values()))}")


probe column:        ID_REF
sample columns:      67
detection p-value:   67  (dropped)
normal samples kept: 56  (11 cell lines dropped)
distinct tissues:    11


In [4]:
# ---- Load the kept columns and relabel ---------------------------------------
# usecols stops the Detection Pval columns from ever being parsed.

df_val = pd.read_csv(data_path, usecols=[probe_col] + keep_cols)

# usecols returns columns in FILE order, not the order requested; reapply the
# (tissue, title) ordering from the previous cell explicitly.
df_val = df_val[[probe_col] + keep_cols]

# ID_REF -> probe_ID, sample titles -> tissue labels. Duplicate column names are
# intentional: one column per replicate, exactly as in the atlas file.
df_val.columns = ["probe_ID"] + [title_to_tissue[c] for c in keep_cols]

print(f"df_val shape (probes × [probe_ID + samples]): {df_val.shape}")


df_val shape (probes × [probe_ID + samples]): (287050, 57)


In [5]:
# ---- Validate the processed matrix -------------------------------------------
betas = df_val.drop(columns="probe_ID")

n_nan    = int(betas.isna().sum().sum())
beta_min = betas.min().min()          # pandas min/max skip NaN
beta_max = betas.max().max()

assert df_val["probe_ID"].is_unique, "Duplicate probe IDs in the validation matrix!"
assert 0.0 <= beta_min and beta_max <= 1.0, \
    f"Beta values out of range: [{beta_min}, {beta_max}]"

print(f"probes:          {df_val.shape[0]}")
print(f"samples:         {betas.shape[1]}")
print(f"probe IDs:       unique — OK")
print(f"beta range:      [{beta_min:.6f}, {beta_max:.6f}] — OK")
print(f"NaN values:      {n_nan}" + ("" if n_nan == 0 else "  ⚠ present"))

# Overlap with the atlas probe universe: informative, not a failure condition.
# Probes are aligned to the deconvolution panel at inference time, not here.
atlas_path   = config.CSV_FILE
atlas_probes = set(pd.read_csv(atlas_path, usecols=["probe_ID"])["probe_ID"])
val_probes   = set(df_val["probe_ID"])

print(f"\natlas probes:    {len(atlas_probes)}")
print(f"shared:          {len(val_probes & atlas_probes)}")
print(f"validation-only: {len(val_probes - atlas_probes)}")
print(f"atlas-only:      {len(atlas_probes - val_probes)}")


probes:          287050
samples:         56
probe IDs:       unique — OK
beta range:      [0.000000, 0.999361] — OK
NaN values:      664  ⚠ present



atlas probes:    296070
shared:          285797
validation-only: 1253
atlas-only:      10273


In [6]:
# ---- Preview the final matrix ------------------------------------------------
# What the written CSV will look like. Compare against the atlas file layout:
# probe_ID first, then one tissue-named column per replicate.

print(f"Final matrix: {df_val.shape[0]} probes × {df_val.shape[1] - 1} samples\n")
print("First 8 probes × first 10 sample columns:")
display(df_val.iloc[:8,:20])

# Replicates and DNA preservation per tissue. FFPE material is chemically degraded
# relative to fresh-frozen and may deconvolve less cleanly. Worth tracking, since
# it is a plausible explanation for any tissue that validates poorly.
summary = (
    normal[normal["title"].isin(keep_cols)]
    .groupby("tissue")
    .agg(
        n_replicates=("title", "size"),
        FF=("sample preservation", lambda s: (s == "FF").sum()),
        FFPE=("sample preservation", lambda s: (s == "FFPE").sum()),
    )
    .sort_index()
)
print("\nReplicates per tissue:")
display(summary)

print(f"\nColumn header sequence ({df_val.shape[1]} columns):")
print("  " + ", ".join(df_val.columns[:14]) + ", ...")


Final matrix: 287050 probes × 56 samples

First 8 probes × first 10 sample columns:


,probe_ID,BM,BM,Brain,Brain,Brain,Breast,Breast,Breast,Breast,Colon,Colon,Liver,Liver,Liver,Lung,Lung,Lung,Lung,Lung
0,cg00101675_BC21,0.679534,0.657813,0.783021,0.844362,0.823294,0.761324,0.790410,0.780788,0.799194,0.848889,0.838895,0.808479,0.779333,0.794850,0.734030,0.748406,0.734975,0.721910,0.741719
1,cg00116289_BC21,0.712968,0.728005,0.737021,0.759064,0.733452,0.730778,0.666464,0.717242,0.708754,0.822184,0.805878,0.848326,0.824714,0.851083,0.798168,0.802375,0.793796,0.783543,0.791859
2,cg00211372_TC21,0.797144,0.806857,0.874401,0.903871,0.805126,0.761552,0.747594,0.742924,0.766542,0.883041,0.847944,0.889135,0.889571,0.825700,0.877004,0.897248,0.854255,0.851341,0.852588
3,cg00531009_BC21,0.817601,0.816956,0.722292,0.736462,0.714000,0.836129,0.808667,0.834932,0.816236,0.841128,0.830830,0.908907,0.853245,0.928615,0.819205,0.849232,0.819917,0.820025,0.860599
4,cg00747726_TC21,0.752966,0.738699,0.825955,0.823125,0.810872,0.738323,0.791828,0.796243,0.715978,0.729522,0.661274,0.857407,0.835940,0.898318,0.666572,0.669278,0.624615,0.614142,0.666002
5,cg00896209_TC21,0.839762,0.813830,0.851278,0.857352,0.850735,0.802191,0.824987,0.827470,0.817419,0.803453,0.806103,0.812842,0.839993,0.780734,0.824708,0.834116,0.839715,0.836115,0.840306
6,cg00910419_BC21,0.033784,0.040947,0.041147,0.046032,0.041165,0.052405,0.057213,0.058053,0.047016,0.035182,0.038556,0.059263,0.044754,0.042233,0.044809,0.044865,0.049597,0.038900,0.040507
7,cg00920372_BC21,0.419995,0.439352,0.497267,0.465421,0.502754,0.298138,0.349300,0.356673,0.368747,0.272867,0.250174,0.849661,0.585731,0.702425,0.350708,0.371547,0.364306,0.323899,0.360380



Replicates per tissue:


,n_replicates,FF,FFPE
tissue,,,
BM,2,2,0
Brain,3,3,0
Breast,4,4,0
Colon,2,2,0
Liver,3,0,3
Lung,14,7,7
Pancreas,4,4,0
Prostate,4,4,0
Skin,7,7,0



Column header sequence (57 columns):
  probe_ID, BM, BM, Brain, Brain, Brain, Breast, Breast, Breast, Breast, Colon, Colon, Liver, Liver, ...


In [ ]:
# ---- Write the processed validation matrix -----------------------------------
# Run this manually once the preview above looks right.

from pathlib import Path

OUT_PATH = config.DATA_DIR / "processed" / "GSE196902_MM285_normal_processed.csv"

# Never write over a source file. Picks up whichever name the metadata path is bound to.
source_paths = [data_path] + [
    p for p in (globals().get("path"), globals().get("meta_path")) if p
]
for src in source_paths:
    assert OUT_PATH.resolve() != Path(src).resolve(), f"Refusing to overwrite source: {src}"

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_val.to_csv(OUT_PATH, index=False)

print(f"Written → DATA_DIR/{OUT_PATH.relative_to(config.DATA_DIR)}")
print(f"  {df_val.shape[0]} probes × {df_val.shape[1] - 1} samples")
print(f"  {OUT_PATH.stat().st_size / 1e6:.1f} MB")
